# 互操作性

(简介-互操作性-关键-要点-1)=
(简介-互操作性-关键-要点-4)=
## 动机

正如 {ref}`analysis frameworks and tools chapter <introduction:analysis-frameworks>`中所讨论的，单细胞分析存在三个主要生态系统：[Bioconductor](https://bioconductor.org/)、[Seurat](https://satijalab.org/seurat/index.html) 和基于 Python 的 [scverse](https://scverse.org/)。
新分析师的一个常见问题是应该关注哪个生态系统。
虽然从其中一个开始是有意义的，并且可以在任何生态系统中进行成功的分析，但有能力的分析师应该熟悉这三者，并能够在它们之间轻松切换。
这使得分析师始终可以使用性能最佳的工具，无论其实施如何。
不习惯切换生态系统的分析师通常会默认使用熟悉的软件包，即使其他地方存在更好的替代方案。
跨生态系统的灵活性还让开发人员能够利用不同语言的优势。
R 对统计建模具有强大的内置支持，而大多数深度学习库都以 Python 为目标。
通过支持常见的磁盘格式和内存数据结构，开发人员可以确保分析师在最合适的平台上访问他们的包。
多生态系统流畅性的另一个动机是数据、结果和文档的可访问性。
数据和结果通常仅以一种格式提供，需要熟悉该生态系统才能访问它们。
在评估使用哪种方法时，还需要对其他生态系统有基本的了解，以阅读文档和教程。

虽然我们鼓励分析师熟悉所有主要生态系统，但只有当它们具有互操作性时才可能在它们之间移动。
值得庆幸的是，在这方面已经做了很多工作，并且在大多数情况下使用标准包现在相对简单。
在本章中，我们讨论通过磁盘或内存在生态系统之间移动数据的各种方式、它们的差异以及优点。
我们专注于单模态数据以及在 R 和 Python 之间移动，因为这些是最常见的情况，但我们也涉及多模态数据和其他语言。

在阅读本章之前，我们首先导入所有必需的 Python 包。

In [ ]:
import tempfile
from pathlib import Path

import anndata2ri
import lamindb as ln
import rpy2.robjects

%load_ext rpy2.ipython

ln.track()

→ connected lamindb: theislab/sc-best-practices
→ created Transform('QaILJZMpZyJ40000'), started new Run('1YEGroVZ...') at 2025-03-26 12:58:46 UTC


## 命名法

因为谈论不同的语言可能会令人困惑，我们尝试使用以下约定：

- **{package}** - R 包
-`package::function()`- R 包中的函数
- **包** - Python 包
-`package.function()`- Python 包中的函数
- **强调** - 其他一些重要概念
-`code`- 代码的其他部分，包括对象、变量等。这也用于文件或目录。

## 基于磁盘的互操作性

在语言之间移动的第一种方法是通过基于磁盘的互操作性。
这涉及用一种语言将文件写入磁盘，然后将该文件读取为第二种语言。
在许多情况下，这种方法比内存互操作性（我们将在下面讨论）更简单、更可靠和可扩展。
尽管如此，它的代价是更大的存储需求和减少的交互性。
当分析的每个阶段都有已建立的流程并且您希望将对象从一个阶段传递到下一个阶段时（特别是作为使用工作流管理器（例如 [Nextflow](https://www.nextflow.io/index.html) 或 [snakemake](https://snakemake.readthedocs.io/en/stable/)）开发的管道的一部分），基于磁盘的互操作性往往效果特别好。
但是，基于磁盘的互操作性对于数据探索或试验方法等交互步骤不太方便，因为每当您想要在语言之间移动时都需要编写新文件。

（简介-互操作性-关键要点-2）=
### 简单格式

在讨论专门为单细胞数据开发的文件格式之前，我们想简要提及常见的简单文本文件格式（例如 CSV、TSV、JSON 等）通常可以作为在语言之间传输数据的答案。
当已经执行了一些分析并且您想要传输的是有关实验的信息的子集时，它们可以很好地工作。
例如，您可能只想传输单元格元数据，但不需要特征元数据、表达式矩阵等。
使用简单文本格式的优点是几乎所有语言都可以很好地支持它们，并且不需要单单元特定的包。
然而，随着您想要传输的内容变得更加复杂，它们很快就会变得不切实际。

### 基于 HDF5 的格式

[Hierarchical Data Format version 5](https://www.hdfgroup.org/solutions/hdf5/) (HDF5) 是用于存储单细胞数据的最常见的开源文件格式。
它专为大型、复杂和异构数据集而设计，使用类似于计算机文件系统的目录式结构。
这允许多种类型的数据以有组织的层次结构存储在单个文件中。虽然 HDF5 非常灵活，但与其交互需要了解文件中数据的结构。
为了标准化此过程，已经制定了用于在 HDF5 文件中存储单细胞数据的具体指南。

#### H5AD

H5AD 格式是 scverse 包使用的`AnnData`对象的 HDF5 磁盘表示形式，通常用于共享单单元数据集。
由于它是 scverse 生态系统的一部分，因此从 Python 读取和写入这些文件得到了良好的支持，并且是 [**anndata** package](https://anndata.readthedocs.io/en/latest/index.html) 核心功能的一部分（了解有关格式 [here](https://anndata.readthedocs.io/en/latest/fileformat-prose.html) 的更多信息）。

为了演示互操作性，我们将加载一个随机生成的小型数据集，该数据集已完成标准分析工作流程的一些步骤以填充各个插槽。

In [ ]:
af = ln.Artifact.connect("theislab/sc-best-practices").get(
    key="introduction/interoperability_adata.h5ad", is_latest=True
)
adata = af.load()
adata

AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

我们将这个模拟对象作为 H5AD 文件写入磁盘，以演示如何从 R 读取这些文件。

In [4]:
temp_dir = tempfile.TemporaryDirectory()
h5ad_file = str(Path(temp_dir.name) / "example.h5ad")

adata.write_h5ad(h5ad_file)

多个 R 包支持读取和写入 H5AD 文件。
但是，它们通常包装 Python **anndata** 包进行文件处理，使用内存中转换步骤在 R 和 Python 之间桥接数据。

##### 用 Bioconductor 读/写 H5AD

[Bioconductor **{zellkonverter}** package](https://bioconductor.org/packages/zellkonverter/) 通过使用 [**{basilisk}** package](https://bioconductor.org/packages/basilisk/) 管理兼容的 Python 环境来简化 H5AD 文件处理。
换句话说，它使 Bioconductor 用户能够无缝地读写 H5AD 文件，而不需要任何 Python 知识。

不幸的是，由于本书的制作方式，我们无法直接运行此处的代码。相反，我们将展示代码以及在 R 会话中运行时的输出：

```r
sce <- zellkonverter::readH5AD(h5ad_file, verbose = TRUE)
```

```
ℹ Using the Python reader
ℹ Using anndata version 0.8.0
✔ Read /.../luke.zappia/Downloads/example.h5ad [113ms]
✔ uns$hvg$flavor converted [17ms]
✔ uns$hvg converted [50ms]
✔ uns$log1p converted [25ms]
✔ uns$neighbors converted [18ms]
✔ uns$pca$params$use_highly_variable converted [16ms]
✔ uns$pca$params$zero_center converted [16ms]
✔ uns$pca$params converted [80ms]
✔ uns$pca$variance converted [17ms]
✔ uns$pca$variance_ratio converted [16ms]
✔ uns$pca converted [184ms]
✔ uns$umap$params$a converted [16ms]
✔ uns$umap$params$b converted [16ms]
✔ uns$umap$params converted [80ms]
✔ uns$umap converted [112ms]
✔ uns converted [490ms]
✔ Converting uns to metadata ... done
✔ X matrix converted to assay [29ms]
✔ layers$counts converted [27ms]
✔ Converting layers to assays ... done
✔ var converted to rowData [25ms]
✔ obs converted to colData [24ms]
✔ varm$PCs converted [18ms]
✔ varm converted [47ms]
✔ Converting varm to rowData$varm ... done
✔ obsm$X_pca converted [15ms]
✔ obsm$X_umap converted [16ms]
✔ obsm converted [80ms]
✔ Converting obsm to reducedDims ... done
ℹ varp is empty and was skipped
✔ obsp$connectivities converted [22ms]
✔ obsp$distances converted [23ms]
✔ obsp converted [92ms]
✔ Converting obsp to colPairs ... done
✔ SingleCellExperiment constructed [164ms]
ℹ Skipping conversion of raw
✔ Converting AnnData to SingleCellExperiment ... done
```

Because we have turned on the verbose output you can see how **{zellkonverter}** reads the file using Python and converts each part of the`AnnData`object to a Bioconductor`SingleCellExperiment`object.我们可以看到结果是什么样的：

```r
sce
```

```
class: SingleCellExperiment
dim: 2000 100
metadata(5): hvg log1p neighbors pca umap
assays(2): X counts
rownames(2000): Gene_0 Gene_1 ... Gene_1998 Gene_1999
rowData names(11): n_cells_by_counts mean_counts ... dispersions_norm
  varm
colnames(100): Cell_0 Cell_1 ... Cell_98 Cell_99
colData names(8): n_genes_by_counts log1p_n_genes_by_counts ...
  pct_counts_in_top_200_genes pct_counts_in_top_500_genes
reducedDimNames(2): X_pca X_umap
mainExpName: NULL
altExpNames(0):
```

然后，任何 Bioconductor 包都可以正常使用该对象。如果我们想写入一个新的 H5AD 文件，我们可以使用`writeH5AD()`函数：

```r
zellkonverter_h5ad_file <- tempfile(fileext = ".h5ad")
zellkonverter::writeH5AD(sce, zellkonverter_h5ad_file, verbose = TRUE)
```

```
ℹ Using anndata version 0.8.0
ℹ Using the 'X' assay as the X matrix
✔ Selected X matrix [29ms]
✔ assays$X converted to X matrix [50ms]
✔ additional assays converted to layers [30ms]
✔ rowData$varm converted to varm [28ms]
✔ reducedDims converted to obsm [68ms]
✔ metadata converted to uns [24ms]
ℹ rowPairs is empty and was skipped
✔ Converting AnnData to SingleCellExperiment ... done
✔ Wrote '/.../.../rj/.../T/.../file102cfa97cc51.h5ad ' [133ms]
```

然后我们可以用Python读取这个文件：

```python
scanpy.read_h5ad(zellkonverter_h5ad_file)
```

```
AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'X_name', 'hvg', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'
```

如果您第一次运行 **{zellkonverter}** 函数，它将创建一个特殊的 Conda 环境，这可能需要一些时间。
创建后，该环境将被重用于后续函数调用。
* *{zellkonverter}** 还提供诸如选择性读取或写入对象的一部分等选项。
有关更多详细信息，请参阅包文档。

[**{sceasy}** package](https://github.com/cellgeni/sceasy) 中提供了将`SingleCellExperiment`对象写入 H5AD 文件的类似功能。
虽然这些包很有效，但包装 Python 会带来一些开销，未来的原生 R H5AD 编写器/读取器可能有助于优化。

##### 用 **{Seurat}** 读/写 H5AD

虽然`h5ad_file`是一个`Path`对象，但我们在本笔记本中使用 R 的方式需要一个字符串，因此我们将其转换为`string`对象。

In [5]:
h5ad_file = str(h5ad_file)

在`Seurat`对象和 H5AD 文件之间进行转换是一个两步过程 [as suggested by this tutorial](https://mojaveazure.github.io/seurat-disk/articles/convert-anndata.html)。
首先，使用 [**{SeuratDisk}** package](https://mojaveazure.github.io/seurat-disk/) 将 H5AD 文件转换为 H5Seurat 文件（`Seurat`对象的自定义 HDF5 格式）。
然后，H5Seurat 文件被读取为`Seurat`对象。

In [6]:
%%R -i h5ad_file

message("Converting H5AD to H5Seurat...")
SeuratDisk::Convert(h5ad_file, dest = "h5seurat", overwrite = TRUE)
message("Reading H5Seurat...")
h5seurat_file <- gsub(".h5ad", ".h5seurat", h5ad_file)
seurat <- SeuratDisk::LoadH5Seurat(h5seurat_file, assays = "RNA")
message("Read Seurat object:")
seurat


R[write to console]: Converting H5AD to H5Seurat...




    an issue that caused a segfault when used with rpy2:
    https://github.com/rstudio/reticulate/pull/1188
    Make sure that you use a version of that package that includes
    the fix.
    

R[write to console]: Registered S3 method overwritten by 'SeuratDisk':
  method            from  
  as.sparse.H5Group Seurat

R[write to console]: 경고:
R[write to console]:  Unknown file type: h5ad

R[write to console]: 경고:
R[write to console]:  'assay' not set, setting to 'RNA'

R[write to console]: Creating h5Seurat file for version 3.1.5.9900

R[write to console]: Adding X as data

R[write to console]: Adding X as counts

R[write to console]: Adding meta.features from var

R[write to console]: Adding X_pca as cell embeddings for pca

R[write to console]: Adding X_umap as cell embeddings for umap

R[write to console]: Adding PCs as feature loadings fpr pca

R[write to console]: Adding miscellaneous information for pca

R[write to console]: Adding standard deviations for pca

R[write to console]: Adding miscellaneous information for umap

R[write to console]: Adding hvg to miscellaneous data

R[write to console]: Adding log1p to miscellaneous data

R[write to console]: Adding layer cou

An object of class Seurat 
2000 features across 100 samples within 1 assay 
Active assay: RNA (2000 features, 0 variable features)
 2 layers present: counts, data
 2 dimensional reductions calculated: pca, umap


请注意，由于结构差异，转换`Seurat`对象比`AnnData`或`SingleCellExperiment`对象更复杂。
有关更多详细信息，请参阅 [conversion function documentation](https://mojaveazure.github.io/seurat-disk/reference/Convert.html)。

* *{sceasy}** 包允许将 H5AD 文件直接读取到`Seurat`或`SingleCellExperiment`对象中。
与依赖于专用 Python 环境的 **{zellkonverter}** 不同，**{sceasy}** 包装了 Python 函数，无需特殊设置。
但是，这意味着您必须手动配置环境，确保 R 可以找到它，并安装必要的软件包。

```r
sceasy_seurat <- sceasy::convertFormat(h5ad_file, from="anndata", to="seurat")
sceasy_seurat
```
```
警告: Feature names cannot have underscores ('_'), replacing with dashes ('-')
X -> counts
An object of class Seurat
2000 features across 100 samples within 1 assay
Active assay: RNA (2000 features, 0 variable features)
 2 dimensional reductions calculated: pca, umap
```

##### 用 **{anndata}** 读/写 H5AD

R [**{anndata}** package](https://anndata.dynverse.org/index.html) 也可用于读取 H5AD 文件。
但是，与上面的包不同，它不会转换为本机 R 对象。
相反，它为 Python 对象提供了一个 R 接口。
这对于访问数据很有用，但很少有分析包会接受它作为输入，因此通常需要进一步的内存中转换。

#### 织机

[Loom file format](http://loompy.org/) 是基于旧版 HDF5 的组学数据规范。
虽然结构与`AnnData and SingleCellExperiment`类似，但它并不依赖于 H5AD 等特定的分析生态系统。
[R](https://github.com/mojaveazure/loomR) 和 [Python](https://pypi.org/project/loompy/) 都提供 Loom 格式支持，也可以通过 [Bioconductor package](https://bioconductor.org/packages/LoomExperiment/) 来写入 Loom 文件。
然而，使用核心生态系统包提供的更高级别的接口通常更方便。
除了数据集共享之外，在使用 [velocyto](http://velocyto.org/) 分析 {ref}`RNA velocity analysis <trajectories:rna-velocity>`的拼接和未拼接读取时，通常会遇到 Loom 文件。

### RDS 文件

您可能会看到用于共享单细胞数据集的另一种文件格式是 RDS 格式。
这是一种用于序列化任意`R`对象的二进制格式（类似于 Python Pickle 文件）。
由于`SingleCellExperiment`和`Seurat`对象并不总是具有匹配的磁盘表示形式，因此 RDS 文件有时用于共享 R 分析的结果。
虽然这在分析项目中是可以的，但由于缺乏与其他生态系统的互操作性，我们不鼓励将其用于公开或与合作者共享数据。
相反，我们建议使用上面提到的可以从多种语言读取的 HDF5 格式之一。

### 新的磁盘格式

虽然基于 HDF5 的格式目前是单细胞数据磁盘表示的标准，但其他较新的技术（例如 [Zarr](https://zarr.dev/) 和 [TileDB](https://tiledb.com/)）具有一些优势，特别是对于非常大的数据集和其他模式。
我们预计将来会为这些格式开发规范，这些规范可能会被社区采用（**anndata** 已经提供了对 Zarr 文件的支持）。

```{admonition} 关键点
- `SingleCellExperiment`：R语言中单细胞数据的Bioconductor标准格式。`AnnData`与`SingleCellExperiment`之间的转换可通过`zellkonverter`完成。
- `Seurat`：R语言中的单细胞分析框架。可通过`SeuratDisk`读写`AnnData`，或通过`sceasy`进行转换。
- `AnnData`：Python中单细胞数据的标准格式。
```

## 内存内互操作性

实现互操作性的第二种方法是处理对象的内存表示。
此方法涉及同时运行的两种编程语言的活动会话，并且从两种编程语言访问同一对象或根据需要在它们之间进行转换。
通常，一种语言作为主要环境，并且有与另一种语言的接口。
这对于交互式分析非常有用，因为它允许分析师同时使用两种语言工作。
创建使用多种语言的文档（例如本书）时也经常使用它。
然而，内存中的互操作性有一些缺点。
分析师必须熟悉这两种环境的设置和使用，复杂的对象可能无法完全支持跨语言，并且数据重复会增加内存开销，使其不太适合大型数据集。

### R 生态系统之间的互操作性

在研究 R 和 Python 之间的内存互操作性之前，让我们考虑一下在两个 R 生态系统之间进行转换的更简单的情况。
* *{Seurat}** 包提供了执行此转换 [as described in this vignette](https://satijalab.org/seurat/articles/conversion_vignette.html) 的函数。

In [7]:
%%R
sce_from_seurat <- Seurat::as.SingleCellExperiment(seurat)
sce_from_seurat


class: SingleCellExperiment 
dim: 2000 100 
metadata(0):
assays(2): counts logcounts
rownames(2000): Gene-0 Gene-1 ... Gene-1998 Gene-1999
rowData names(0):
colnames(100): Cell_0 Cell_1 ... Cell_98 Cell_99
colData names(9): n_genes_by_counts log1p_n_genes_by_counts ...
  pct_counts_in_top_500_genes ident
reducedDimNames(2): PCA UMAP
mainExpName: RNA
altExpNames(0):


In [8]:
%%R
seurat_from_sce <- Seurat::as.Seurat(sce_from_seurat)
seurat_from_sce


An object of class Seurat 
2000 features across 100 samples within 1 assay 
Active assay: RNA (2000 features, 0 variable features)
 2 layers present: counts, data
 2 dimensional reductions calculated: PCA, UMAP


这里的困难部分是由于两个对象的结构之间的差异。
确保正确设置参数非常重要，以便转换函数知道要转换哪些信息以及将其放置在何处。

在许多情况下，可能不需要将`Seurat`对象转换为`SingleCellExperiment`。
这是因为许多用于单细胞分析的核心 Bioconductor 包也被设计为接受矩阵作为输入。

In [9]:
%%R
# Calculate Counts Per Million using the Bioconductor scuttle package
# with a matrix in a Seurat object
cpm <- scuttle::calculateCPM(Seurat::GetAssayData(seurat, slot = "counts"))
cpm[1:10, 1:10]


10 x 10 sparse Matrix of class "dgCMatrix"


R[write to console]:   [[ suppressing 10 column names ‘Cell_0’, ‘Cell_1’, ‘Cell_2’ ... ]]



                                                                      
Gene-0 594.0456    .     984.0238  610.329 964.4394 616.4169    .     
Gene-1 594.0456 1168.519 622.6029    .     608.7911   .         .     
Gene-2   .         .     984.0238    .     608.7911 973.2674 1205.5970
Gene-3 594.0456    .     622.6029  610.329 964.4394 616.4169    .     
Gene-4 594.0456    .     622.6029  966.820 964.4394   .         .     
Gene-5 594.0456  580.157 622.6029  610.329 608.7911   .       601.7422
Gene-6   .         .     622.6029  610.329   .      616.4169  601.7422
Gene-7 594.0456  580.157 622.6029  966.820 608.7911 616.4169    .     
Gene-8 594.0456  580.157   .      1219.608 608.7911 973.2674    .     
Gene-9 942.9097    .       .         .     964.4394   .       954.8014
                                    
Gene-0  576.0808  619.7236  604.3937
Gene-1  918.4111 1234.7593  958.4625
Gene-2    .       619.7236 1209.8233
Gene-3    .       619.7236  958.4625
Gene-4  576.0808  979.8761  604.39

但是，重要的是要确保您访问正确的信息并将所有结果存储在正确的位置（如果需要）。

### 从 Python 访问 R

The Python R 接口由 [**rpy2** package](https://rpy2.github.io/doc/latest/html/index.html) 提供。
这允许您从 Python 访问 R 函数和对象。
例如：

In [10]:
counts_mat = adata.layers["counts"].T.toarray()

with rpy2.robjects.conversion.localconverter(rpy2.robjects.numpy2ri.converter):
    rpy2.robjects.globalenv["counts_mat"] = counts_mat

cpm = rpy2.robjects.r("scuttle::calculateCPM(counts_mat)")
cpm

494.804552,494.804552,0.000000,...,519.750520,519.750520,0.000000


Common Python 对象（列表、矩阵、`DataFrame`s 等）也可以传递给 R。

如果您使用的是 Jupyter 笔记本（正如我们在本书中使用的那样），您可以使用 IPython 魔法接口来使用本机 R 代码创建单元格（根据需要传递对象）。
例如，以`%%R -i input -o output`启动单元表示将`input`作为输入，运行 R 代码，然后返回`output`作为输出。

In [11]:
%%R -i counts_mat -o magic_cpm
# R code running using IPython magic
magic_cpm <- scuttle::calculateCPM(counts_mat)


In [12]:
# Python code accessing the results
magic_cpm

array([[ 494.8045522 ,    0.        , 1027.2213662 , ...,    0.        ,
           0.        ,  519.75051975],
       [ 494.8045522 , 1445.78313253,  513.6106831 , ...,    0.        ,
         499.5004995 ,  519.75051975],
       [   0.        ,    0.        , 1027.2213662 , ...,  485.90864917,
         499.5004995 ,    0.        ],
       ...,
       [ 494.8045522 ,    0.        ,  513.6106831 , ...,    0.        ,
           0.        ,  519.75051975],
       [ 989.6091044 ,  481.92771084,    0.        , ...,    0.        ,
         499.5004995 ,  519.75051975],
       [2474.02276101,  963.85542169,  513.6106831 , ...,  485.90864917,
         999.000999  ,    0.        ]])

这是您在后面的章节中最常看到的方法。
有关使用 **rpy2** 的更多信息，请参阅 [the documentation](https://rpy2.github.io/doc/latest/html/index.html)。

要以这种方式处理单细胞数据，[**anndata2ri** package](https://icb-anndata2ri.readthedocs-hosted.com/en/latest/) 特别有用。
作为 **rpy2** 的扩展，它允许 R 将`AnnData`对象识别为`SingleCellExperiment`对象，从而消除不必要的转换并实现 R 代码在 Python 对象上的无缝执行。
此外，它还有助于稀疏 **scipy** 矩阵的转换。

在此示例中，为了将 Python 会话中的`AnnData`对象传递给 R，我们必须首先将其转换为`SingleCellExperiment`。

In [13]:
with rpy2.robjects.conversion.localconverter(anndata2ri.converter):
    r_adata = rpy2.robjects.conversion.py2rpy(adata)

现在，我们可以将它传递给 R。

In [14]:
%%R -i r_adata
qc <- scuttle::perCellQCMetrics(r_adata)
head(qc)

DataFrame with 6 rows and 3 columns
             sum  detected     total
       <numeric> <integer> <numeric>
Cell_0      2021      1297      2021
Cell_1      2075      1314      2075
Cell_2      1947      1233      1947
Cell_3      1986      1250      1986
Cell_4      1987      1255      1987
Cell_5      1930      1266      1930


请注意，如果对象（或其一部分）无法正确连接（例如，如果存在不受支持的数据类型），您仍然会遇到问题。
在这种情况下，您可能需要先修改对象，然后才能访问它。

### Accessing Python 来自 R

R 会话中的 Accessing Python 与从 Python 访问 R 类似，但此处的接口由 [**{reticulate}** package](https://rstudio.github.io/reticulate/) 提供。
一旦加载，我们就可以从 R 访问 Python 函数和对象。

In [15]:
%%R
reticulate_list <- reticulate::r_to_py(LETTERS)
print(reticulate_list)
py_builtins <- reticulate::import_builtins()
py_builtins$zip(letters, LETTERS)


List (26 items)


如果您正在处理 [RMarkdown](https://rmarkdown.rstudio.com/) 或 [Quarto](https://quarto.org/) 文档，您还可以使用 **{reticulate}** Python 引擎编写本机 Python 块。
当我们这样做时，我们可以使用神奇的`r`和`py`变量来访问其他语言的对象（以下代码是未运行的示例）。

````
```{r}
# 访问 Python 对象的 R 块
打印（py$py_object）
```

```{python}
# 访问 R 对象的 Python 块
打印（r$r_object）
```
````

与 **anndata2ri** 不同，没有 R 包为 Python 提供将`SingleCellExperiment`或`Seurat`对象视为`AnnData`对象的直接接口。
但是，我们仍然可以使用 **{reticulate}** 访问`AnnData`的大部分部分（此代码未运行）。

```r
# Print an AnnData object in a Python environment
py$adata
```
```
AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'
```
```r
# Alternatively use the Python anndata package to read a H5AD file
anndata <- reticulate::import("anndata")
anndata$read_h5ad(h5ad_file)
```
```
AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'
```
```r
# Access the obs slot, pandas DataFrames are automatically converted to R data.frames
head(adata$obs)
```
```
       n_genes_by_counts log1p_n_genes_by_counts total_counts
Cell_0              1246                7.128496         1965
Cell_1              1262                7.141245         2006
Cell_2              1262                7.141245         1958
Cell_3              1240                7.123673         1960
Cell_4              1296                7.167809         2027
Cell_5              1231                7.116394         1898
       log1p_total_counts pct_counts_in_top_50_genes
Cell_0           7.583756                  10.025445
Cell_1           7.604396                   9.521436
Cell_2           7.580189                   9.959142
Cell_3           7.581210                   9.183673
Cell_4           7.614805                   9.718796
Cell_5           7.549083                  10.168599
       pct_counts_in_top_100_genes pct_counts_in_top_200_genes
Cell_0                    17.65903                    30.89059
Cell_1                    16.99900                    29.71087
Cell_2                    17.62002                    30.28601
Cell_3                    16.83673                    30.45918
Cell_4                    17.11889                    30.04440
Cell_5                    18.07165                    30.29505
       pct_counts_in_top_500_genes
Cell_0                    61.42494
Cell_1                    59.62114
Cell_2                    60.92952
Cell_3                    61.07143
Cell_4                    59.64480
Cell_5                    61.48577
```

如上所述，R **{anndata}** 包为`AnnData`对象提供了 R 接口，但目前许多分析包并未使用它。

对于需要处理整个对象的更复杂的分析，可能需要在 R 和 Python
 While 之间完全转换对象，这种方法由于数据重复而内存效率不高，但它允许访问更广泛的包。

* *{zellkonverter}** 包提供了用于此转换的函数。
与读取 H5AD 文件的功能不同，此过程使用标准 Python 环境，而不是专门创建的环境（代码未运行）。


```r
# Convert an AnnData to a SingleCellExperiment
sce <- zellkonverter::AnnData2SCE(adata, verbose = TRUE)
sce
```
```
✔ uns$hvg$flavor converted [21ms]
✔ uns$hvg converted [62ms]
✔ uns$log1p converted [22ms]
✔ uns$neighbors converted [21ms]
✔ uns$pca$params$use_highly_variable converted [22ms]
✔ uns$pca$params$zero_center converted [31ms]
✔ uns$pca$params converted [118ms]
✔ uns$pca$variance converted [17ms]
✔ uns$pca$variance_ratio converted [17ms]
✔ uns$pca converted [224ms]
✔ uns$umap$params$a converted [15ms]
✔ uns$umap$params$b converted [17ms]
✔ uns$umap$params converted [80ms]
✔ uns$umap converted [115ms]
✔ uns converted [582ms]
✔ Converting uns to metadata ... done
✔ X matrix converted to assay [44ms]
✔ layers$counts converted [29ms]
✔ Converting layers to assays ... done
✔ var converted to rowData [37ms]
✔ obs converted to colData [23ms]
✔ varm$PCs converted [18ms]
✔ varm converted [49ms]
✔ Converting varm to rowData$varm ... done
✔ obsm$X_pca converted [17ms]
✔ obsm$X_umap converted [17ms]
✔ obsm converted [80ms]
✔ Converting obsm to reducedDims ... done
ℹ varp is empty and was skipped
✔ obsp$connectivities converted [21ms]
✔ obsp$distances converted [22ms]
✔ obsp converted [89ms]
✔ Converting obsp to colPairs ... done
✔ SingleCellExperiment constructed [241ms]
ℹ Skipping conversion of raw
✔ Converting AnnData to SingleCellExperiment ... done
class: SingleCellExperiment
dim: 2000 100
metadata(5): hvg log1p neighbors pca umap
assays(2): X counts
rownames(2000): Gene_0 Gene_1 ... Gene_1998 Gene_1999
rowData names(11): n_cells_by_counts mean_counts ... dispersions_norm
  varm
colnames(100): Cell_0 Cell_1 ... Cell_98 Cell_99
colData names(8): n_genes_by_counts log1p_n_genes_by_counts ...
  pct_counts_in_top_200_genes pct_counts_in_top_500_genes
reducedDimNames(2): X_pca X_umap
mainExpName: NULL
altExpNames(0):
```

反过来也可以进行同样的操作：

```r
adata2 <- zellkonverter::SCE2AnnData(sce, verbose = TRUE)
adata2
```
```
ℹ Using the 'X' assay as the X matrix
✔ Selected X matrix [27ms]
✔ assays$X converted to X matrix [38ms]
✔ additional assays converted to layers [31ms]
✔ rowData$varm converted to varm [15ms]
✔ reducedDims converted to obsm [63ms]
✔ metadata converted to uns [23ms]
ℹ rowPairs is empty and was skipped
✔ Converting AnnData to SingleCellExperiment ... done
AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'X_name', 'hvg', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'
```

## 多模式数据的互操作性

多模式数据的复杂性给互操作性带来了额外的挑战。
`SingleCellExperiment`（通过“替代实验”，必须共享相同的细胞列尺寸）和`Seurat`（使用“测定”）都支持多种模式。
但是，`AnnData`仅限于单峰数据。

为了解决此限制，开发了`MuData`对象（在 [analysis frameworks and tools chapter]({ref}`analysis frameworks and tools chapter <introduction:analysis-frameworks>`) 中引入）作为多模式数据集`AnnData`的扩展。
开发人员在设计中考虑了互操作性。
虽然 MuData 的主要平台是 Python，但作者提供了 [MuDataSeurat R package](https://pmbio.github.io/MuDataSeurat/) 用于将磁盘上的 H5MU 格式读取为`Seurat`对象，以及 [MuData R package](https://bioconductor.org/packages/MuData/) 用于对 Bioconductor`MultiAssayExperiment`对象执行相同操作。这个官方支持非常有用，但由于对象之间的差异，仍然存在一些不一致的情况。 MuData 作者还提供了`AnnData`和`MuData`的 [Julia implementation](https://docs.juliahub.com/Muon/QfqCh/0.1.1/objects/)。

下面是使用 Python 和 R 包读取和写入小型示例`MuData`数据集的示例。

为了解决这个问题，在 [analysis frameworks and tools chapter]({ref}`analysis frameworks and tools chapter <introduction:analysis-frameworks>`) 中引入的`MuData`对象为多模式数据集扩展了`AnnData`。
`MuData`的设计考虑到了互操作性，主要是一个基于 Python 的框架，但作者提供了 [MuDataSeurat R package](https://pmbio.github.io/MuDataSeurat/)。
这样可以将磁盘上的 H5MU 格式读取为`Seurat`对象，而 [MuData R package](https://bioconductor.org/packages/MuData/) 允许转换为`MultiAssayExperiment`对象。
虽然这个官方支持非常有用，但由于对象之间的差异，仍然存在一些不一致的情况。
`MuData`作者还提供了`AnnData`和`MuData`的 [Julia implementation](https://docs.juliahub.com/Muon/QfqCh/0.1.1/objects/)。

下面是使用 Python 和 R 包读取和写入小型示例`MuData`数据集的示例。

### Python

In [ ]:
# Read file
af_mudata = ln.Artifact.connect("theislab/sc-best-practices").get(
    key="introduction/interoperability_mdata.h5mu", is_latest=True
)
mdata = af_mudata.load()
mdata

/Users/seohyon/miniconda3/envs/interoperability/lib/python3.12/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/Users/seohyon/miniconda3/envs/interoperability/lib/python3.12/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


MuData object with n_obs × n_vars = 1000 × 150
  var:	'dummy_var'
  2 modalities
    A:	1000 x 100
      obs:	'dummy_obs'
      var:	'dummy_var'
    B:	1000 x 50
      obs:	'dummy_obs'
      var:	'dummy_var'


### R

#### Bioconductor

从`MultiAssayExperiment`对象读取/写入

In [36]:
import shutil
from pathlib import Path

# Save MuData file locally for R
Path("data").mkdir(parents=True, exist_ok=True)

local_path = af_mudata.cache().path

target_path = Path("data") / "interoperability_mdata.h5mu"
shutil.copy(local_path, target_path)
print(f"File copied to: {target_path}")

File copied to: data/interoperability_mdata.h5mu


In [37]:
%%R
mae <- MuData::readH5MU("data/interoperability_mdata.h5mu")
print(mae)

bioc_h5mu_file <- tempfile(fileext = ".h5mu")
MuData::writeH5MU(mae, bioc_h5mu_file)


A MultiAssayExperiment object of 2 listed
 experiments with user-defined names and respective classes.
 Containing an ExperimentList class object of length 2:
 [1] A: SingleCellExperiment with 100 rows and 1000 columns
 [2] B: SingleCellExperiment with 50 rows and 1000 columns
Functionality:
 experiments() - obtain the ExperimentList instance
 colData() - the primary/phenotype DataFrame
 sampleMap() - the sample coordination DataFrame
 `$`, `[`, `[[` - extract colData columns, subset, or experiment
 *Format() - convert into a long or wide DataFrame
 assays() - convert ExperimentList to a SimpleList of matrices
 exportClass() - save data to flat files


#### Seurat

从`Seurat`对象读取/写入

In [32]:
%%R
seurat <- MuDataSeurat::ReadH5MU("data/interoperability_mdata.h5mu")
print(seurat)

seurat_h5mu_file <- tempfile(fileext = ".h5mu")
MuDataSeurat::WriteH5MU(seurat, seurat_h5mu_file)


An object of class Seurat 
150 features across 1000 samples within 2 assays 
Active assay: A (100 features, 0 variable features)
 2 layers present: counts, data
 1 other assay present: B


## 与其他语言的互操作性

在这里，我们简要列出了一些用于单细胞数据与 R 和 Python 以外的语言的互操作性的资源和工具。

### 朱莉娅

- [Muon.jl](https://docs.juliahub.com/Muon/QfqCh/0.1.1/objects/) 提供`AnnData`和`MuData`对象的 Julia 实现，以及 H5AD 和 H5MU 格式的 IO
- [scVI.jl](https://github.com/maren-ha/scVI.jl) 提供`AnnData`的 Julia 实现以及 H5AD 格式的 IO

### JavaScript

- [Vitessce](http://vitessce.io/) 包含来自使用 Zarr 格式存储的`AnnData`对象的加载程序
- [kana family](https://github.com/jkanche/kana) 支持读取保存为 RDS 文件的 H5AD 文件和`SingleCellExperiment`对象

### 铁锈

- [anndata-rs](https://github.com/kaizhang/anndata-rs) 提供 AnnData 的 Rust 实现以及对 H5AD 格式的高级 IO 支持


## 会话信息

## Python

In [33]:
import session_info

session_info.show()

/Users/seohyon/miniconda3/envs/interoperability/lib/python3.12/site-packages/session_info/main.py:213: UserWarning: The '__version__' attribute is deprecated and will be removed in MarkupSafe 3.1. Use feature detection, or `importlib.metadata.version("markupsafe")`, instead.
  mod_version = _find_version(mod.__version__)


## R

In [34]:
%%R
sessioninfo::session_info()


─ Session info ───────────────────────────────────────────────────────────────
 setting  value
 version  R version 4.3.3 (2024-02-29)
 os       macOS 15.3.2
 system   x86_64, darwin13.4.0
 ui       unknown
 language (EN)
 collate  C
 ctype    UTF-8
 tz       Europe/Berlin
 date     2025-03-26
 pandoc   3.6.3 @ /Users/seohyon/miniconda3/envs/interoperability/bin/pandoc
 quarto   NA

─ Packages ───────────────────────────────────────────────────────────────────
 package              * version    date (UTC) lib source
 abind                  1.4-8      2024-09-12 [1] CRAN (R 4.3.3)
 beachmat               2.18.0     2023-10-24 [1] Bioconductor
 Biobase              * 2.62.0     2023-10-24 [1] Bioconductor
 BiocGenerics         * 0.48.1     2023-11-01 [1] Bioconductor
 BiocParallel           1.36.0     2023-10-24 [1] Bioconductor
 bit                    4.6.0      2025-03-06 [1] CRAN (R 4.3.3)
 bit64                  4.6.0-1    2025-01-16 [1] CRAN (R 4.3.3)
 bitops                 1.0-9   

## 参考

```{bibliography}
:filter: docname in docnames
:labelprefix: int
```

## 贡献者

我们衷心感谢以下人员的贡献：

### 作者

* Luke Zappia
* Seo H. Kim

### 审稿人

* Lukas Heumos
* Isaac Virshup
* Anastasia Litinetskaya
* Ludwig Geistlinger
* Peter Hickey